In [ ]:


import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------------------------------
# SETTINGS
# -----------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SHOT_LIST = [1, 3, 5, 10, 20, 30, 40, 50]

results = []

# -----------------------------------------------------
# LOOP OVER SHOTS
# -----------------------------------------------------

for N_SHOT in SHOT_LIST:

    print(f"\nRunning {N_SHOT}-Shot Evaluation")

    support_features = []
    support_labels = []

    classes = torch.unique(train_labels)

    # -----------------------------------------
    # SELECT SUPPORT SET
    # -----------------------------------------

    for cls in classes:

        idx = torch.where(
            train_labels == cls
        )[0]

        # safety check
        n = min(
            N_SHOT,
            len(idx)
        )

        idx = idx[
            torch.randperm(
                len(idx)
            )[:n]
        ]

        support_features.append(
            train_features[idx]
        )

        support_labels.append(
            train_labels[idx]
        )

    support_features = torch.cat(
        support_features
    )

    support_labels = torch.cat(
        support_labels
    )

    # -----------------------------------------
    # PROTOTYPES
    # -----------------------------------------

    prototypes = []

    for cls in classes:

        proto = support_features[
            support_labels == cls
        ].mean(0)

        prototypes.append(proto)

    prototypes = torch.stack(
        prototypes
    )

    # -----------------------------------------
    # NORMALIZATION
    # -----------------------------------------

    test_norm = F.normalize(
        test_features,
        dim=1
    )

    proto_norm = F.normalize(
        prototypes,
        dim=1
    )

    # -----------------------------------------
    # DISTANCE
    # -----------------------------------------

    distances = torch.cdist(
        test_norm,
        proto_norm
    )

    preds = torch.argmin(
        distances,
        dim=1
    )

    y_true = test_labels.numpy()
    y_pred = preds.numpy()

    # -----------------------------------------
    # METRICS
    # -----------------------------------------

    acc = accuracy_score(
        y_true,
        y_pred
    )

    prec = precision_score(
        y_true,
        y_pred,
        average="weighted"
    )

    rec = recall_score(
        y_true,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    results.append([
        N_SHOT,
        acc,
        prec,
        rec,
        f1
    ])

# -----------------------------------------------------
# RESULTS TABLE
# -----------------------------------------------------

results_df = pd.DataFrame(
    results,
    columns=[
        "Shot",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ]
)

# convert to percentage
for col in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score"
]:
    results_df[col] = (
        results_df[col] * 100
    ).round(4)

# -----------------------------------------------------
# DISPLAY
# -----------------------------------------------------

print("\n")
print("="*70)
print("SHOT SENSITIVITY ANALYSIS")
print("="*70)

display(results_df)

# -----------------------------------------------------
# SAVE CSV
# -----------------------------------------------------

results_df.to_csv(
    "shot_analysis_results.csv",
    index=False
)

print(
    "\nSaved: shot_analysis_results.csv"
)

# -----------------------------------------------------
# PLOT : ACCURACY
# -----------------------------------------------------

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    results_df["Shot"],
    results_df["Accuracy"],
    marker="o"
)

plt.xlabel("Number of Shots")
plt.ylabel("Accuracy (%)")

plt.title(
    "Few-Shot Performance Analysis"
)

plt.grid(True)

plt.tight_layout()

plt.savefig(
    "shot_accuracy_curve.png",
    dpi=300
)

plt.show()

print(
    "\nSaved: shot_accuracy_curve.png"
)